# Module 3: What Predicts Prospect Translation?

**Question:** Which pitch-level metrics most separate prospects who convert tools into production from those who don't?

With a 43-prospect cohort split into small outcome groups (9 stars vs 18 disappointing/bust), a traditional ML classifier can't reliably discriminate. Instead, we use **effect size analysis** (Cohen's d) — a more honest approach that quantifies how much each metric differs between stars and busts without overfitting to noise.

We also run XGBoost with leave-one-out CV as a sanity check, but the effect sizes are the primary finding.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="urllib3")

import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from fire_fishman.data.statcast import get_statcast_pitches, get_batting_stats
from fire_fishman.data.prospects import get_prospect_df
from fire_fishman.features.tools_score import compute_tools_for_cohort
from fire_fishman.features.pitch_level import compute_pitch_features_for_cohort

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 7)

In [ ]:
pitches_2023 = get_statcast_pitches(2023)
pitches_2024 = get_statcast_pitches(2024)
pitches = pd.concat([pitches_2023, pitches_2024], ignore_index=True)

prospects = get_prospect_df()
batter_ids = prospects["mlbam_id"].tolist()

tools_df = compute_tools_for_cohort(pitches, batter_ids)
pitch_features = compute_pitch_features_for_cohort(pitches, batter_ids)

feature_df = tools_df.join(pitch_features, how="inner")
feature_df = feature_df.join(prospects.set_index("mlbam_id")[["name", "outcome", "pre_debut_rank"]])
feature_df["success"] = feature_df["outcome"].isin(["star", "solid"]).astype(int)

print(f"Prospect cohort: {len(feature_df)} players")
print(f"Stars/Solid: {feature_df['success'].sum()} | Disappointing/Bust: {(~feature_df['success'].astype(bool)).sum()}")

## 1. Effect Size Analysis (Cohen's d)

For each pitch-level metric, how large is the difference between stars and busts? Cohen's d > 0.5 is a medium effect, > 0.8 is large.

This is more appropriate than ML for a cohort this small — it tells us which metrics have the clearest signal without overfitting.

In [ ]:
stars = feature_df[feature_df["outcome"] == "star"]
failures = feature_df[feature_df["outcome"].isin(["disappointing", "bust"])]

key_metrics = [
    "chase_rate", "whiff_rate", "zone_contact_rate",
    "whiff_rate_fastball", "whiff_rate_breaking", "whiff_rate_offspeed",
    "whiff_rate_elite_96plus", "whiff_rate_two_strike",
    "chase_rate_breaking", "chase_rate_offspeed",
]

effects = []
print(f"{'Metric':<30} {'Stars':>8} {'Busts':>8} {'Diff':>8} {'Cohen d':>8}")
print("-" * 70)

for metric in key_metrics:
    s = stars[metric].dropna()
    f = failures[metric].dropna()
    if len(s) < 2 or len(f) < 2:
        continue
    pooled_std = np.sqrt((s.std()**2 + f.std()**2) / 2)
    d = (s.mean() - f.mean()) / pooled_std if pooled_std > 0 else 0
    effects.append((metric, s.mean(), f.mean(), s.mean() - f.mean(), d))
    flag = "***" if abs(d) > 0.8 else "**" if abs(d) > 0.5 else "*" if abs(d) > 0.2 else ""
    print(f"  {metric:<28} {s.mean():>7.1%} {f.mean():>7.1%} {s.mean()-f.mean():>+7.1%} {d:>7.2f} {flag}")

print("\n  * small effect  ** medium effect  *** large effect")

In [ ]:
# Visualize effect sizes
effects_df = pd.DataFrame(effects, columns=["metric", "star_mean", "bust_mean", "diff", "cohens_d"])
effects_df = effects_df.sort_values("cohens_d", ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#2ecc71" if d < 0 else "#e74c3c" for d in effects_df["cohens_d"]]
ax.barh(effects_df["metric"].str.replace("_", " ").str.title(),
        effects_df["cohens_d"], color=colors, alpha=0.85)
ax.axvline(0, color="black", linewidth=0.5)
ax.axvline(-0.5, color="gray", linestyle="--", alpha=0.3)
ax.axvline(0.5, color="gray", linestyle="--", alpha=0.3)
ax.set_xlabel("Cohen's d (Stars vs Disappointing/Bust)\n← Stars are lower (better) | Stars are higher →", fontsize=11)
ax.set_title("What Separates Stars from Busts?\nEffect size of each pitch-level metric", fontsize=14)

plt.tight_layout()
plt.savefig("../outputs/figures/effect_sizes.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Key Insight

The three clearest current effect sizes are whiff-based rather than chase-based:

1. **Whiff rate vs 96+ mph** (d = -0.46): Stars handle elite velocity better.
2. **Breaking-ball whiff rate** (d = -0.44): Stars make more contact when they commit to breaking pitches.
3. **Two-strike whiff rate** (d = -0.43): Stars protect the zone better when the at-bat is on the line.

Notably, **chase rates** separate far less at the cohort level (overall chase d = +0.05, breaking-ball chase d = -0.10, offspeed chase d = +0.16). The chase-rate collapse documented in Notebooks 02, 05, and 06 is a Yankees case-study pattern rather than a cohort-wide separator. The aggregate metrics still hide the strongest signal: velocity handling and pitch-specific whiff rates.

## 3. XGBoost Sanity Check

We run XGBoost with LOO-CV to see if a classifier can discriminate. With n=43 and small outcome groups, we expect it to struggle — this is a check, not the primary analysis.

In [ ]:
try:
    from sklearn.model_selection import LeaveOneOut
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import accuracy_score, roc_auc_score
    from xgboost import XGBClassifier
    
    exclude_cols = ["name", "outcome", "success", "total_pitches_seen"]
    exclude_cols += [c for c in feature_df.columns if c.endswith("_z")]
    feature_cols = [c for c in feature_df.columns
                    if c not in exclude_cols and feature_df[c].dtype in ["float64", "int64", "int32"]]
    
    X = feature_df[feature_cols].copy()
    y = feature_df["success"].values
    
    imputer = SimpleImputer(strategy="median")
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
    
    loo = LeaveOneOut()
    predictions, probabilities = [], []
    
    for train_idx, test_idx in loo.split(X_imputed):
        model = XGBClassifier(n_estimators=50, max_depth=2, learning_rate=0.1,
                              min_child_weight=3, subsample=0.8, eval_metric="logloss", verbosity=0)
        model.fit(X_imputed.iloc[train_idx], y[train_idx])
        predictions.append(model.predict(X_imputed.iloc[test_idx])[0])
        probabilities.append(model.predict_proba(X_imputed.iloc[test_idx])[0, 1])
    
    acc = accuracy_score(y, predictions)
    print(f"LOO-CV Accuracy: {acc:.1%} (base rate: {y.mean():.1%})")
    print(f"\nAs expected, the classifier doesn't outperform the base rate with n={len(feature_df)}.")
    print(f"This is why we use effect sizes as the primary analysis — they're more honest")
    print(f"about what the data can support with this sample size.")
except Exception as e:
    print(f"XGBoost sanity check skipped: {e}")
    print("This is a sanity check only — effect sizes above are the primary analysis.")


## 4. Yankees Prospect Profiles

Where do Volpe and Dominguez sit on the metrics that matter most?

In [ ]:
top_metrics = ["whiff_rate_elite_96plus", "whiff_rate_breaking", "whiff_rate_two_strike"]
star_benchmarks = {col: stars[col].mean() for col in top_metrics}

for name in ["Anthony Volpe", "Jasson Dominguez"]:
    row = feature_df[feature_df["name"] == name]
    if len(row) == 0:
        continue
    print(f"\n{'='*50}")
    print(f"  {name.upper()}")
    print(f"{'='*50}")
    for col in top_metrics:
        current = row[col].values[0]
        target = star_benchmarks[col]
        gap = current - target
        label = col.replace('_', ' ').title()
        flag = " ⚠️" if gap > 0.03 else " ✓" if gap < 0.01 else ""
        print(f"  {label:<30} {current:.1%} (star avg: {target:.1%}, gap: {gap:+.1%}){flag}")

## Takeaway

The signal is in the **specifics**, not the aggregates. Overall whiff rate is less informative than the specific failure modes. How hitters handle 96+ mph fastballs, breaking-ball contact, and two-strike protection is where outcome groups diverge most clearly in the current cohort.

→ Continue to **Notebook 04** for actionable development prescriptions.